# 12. Lecke - Csevegési Előzmények Csökkentése Agent Scratchpaddel

Ez a jegyzetfüzet bemutatja, hogyan lehet kezelni a kontextust hosszú beszélgetések során a Microsoft Agent Framework segítségével. Ahogy a beszélgetések nőnek, a tokenek száma is növekszik — végül meghaladja a modell kontextusablakát. Ezt egy **kontextus összefoglaló mintával** és egy **agent scratchpaddel** oldjuk meg a tartós memória érdekében.

## Amit megtanulsz:
1. **Miért fontos a kontextus kezelése**: A token limit és kontextusablak megértése
2. **Kontextus-Tudatos ügynökök**: Ügynökök építése, amelyek kezelik a saját beszélgetéses kontextusukat
3. **Kontextus összefoglaló minta**: Eszközök használata a beszélgetési előzmények tömörítésére
4. **Agent Scratchpad**: Tartós memória, amely túléli a kontextus csökkentését

## Előfeltételek:
- Azure OpenAI környezet beállítása környezeti változókkal
- Az alapvető ügynök fogalmak megértése az előző leckékből


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Miért fontos a kontextus kezelése

Minden LLM-nek van egy véges **kontekstus ablak**-a — a maximális token-szám, amit egyetlen kérésben feldolgozni tud. Ahogy egy többszörös fordulós beszélgetés halad előre:

- A **tokenek száma lineárisan növekszik** minden felhasználói üzenettel és asszisztens válasszal.
- A **prompt tokenek dominálják a költséget**, mert az egész előzményt minden fordulóban újra elküldik.
- Végül a beszélgetés **túlhaladja a kontextus ablakot**, és a modell vagy levágja a szöveget, vagy hibát jelez.

### Stratégiák a kontextus kezelésére

| Stratégia | Működése | Kompromisszum |
|---|---|---|
| **Levágás** | A legrégebbi üzenetek eltávolítása | Korai kontextus elvesztése |
| **Összefoglalás** | Régebbi üzenetek tömörítése összefoglalóvá | Néhány részlet elveszik, de a kulcspontok megmaradnak |
| **Jegyzetfüzet / Külső memória** | Kulcsinformációk tárolása a beszélgetésen kívül | Eszközhívásokat igényel, de bármilyen csökkentést túlél |

Ebben a jegyzetfüzetben az **összefoglalást** és egy **jegyzetfüzet eszközt** kombinálunk, hogy az ügynök folytonosságot tarthasson fenn még akkor is, ha a beszélgetési előzmény tömörítésre kerül.


## Kontextusérzékeny ügynök létrehozása


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Hosszú Beszélgetés Szimulálása

Nézzünk meg egy többfordulós beszélgetést, hogy lássuk, hogyan halmozódik a kontextus. Az ügynöknek meg kell őriznie a kulcsfontosságú részleteket (preferenciák, költségvetés, utazási dátumok) a fordulók során, és folytonosságot kell mutatnia.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Figyeld meg, hogyan őrzi meg az ügynök a korábbi fordulók kontextusát — tud Japánról, szushiról, templomokról, fotózásról, a 3000 dolláros költségvetésről, egyéni utazásról és az áprilisi időszakról. Egy rövid beszélgetésben ez jól működik, de ahogy a beszélgetés nő, az egész előzmény újbóli elküldése költséges lesz.

Folytassuk a beszélgetést további fordulókkal, hogy lássuk a kontextus felhalmozódását:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Kontextus Összefoglalási Minta

Ahogy a beszélgetés nő, használhatunk egy **összefoglaló eszközt**, hogy a felgyülemlett kontextust tömör formában rögzítsük. Az ügynök ezt az eszközt hívja meg, hogy rögzítse a fontos preferenciákat, így még ha a régebbi üzenetek el is vesznek, a lényeges információk megmaradnak.

Ez a minta az alapja a kifinomultabb előzmény csökkentésnek:
1. Az ügynök azonosítja a beszélgetés kulcsfontosságú tényeket
2. Meghívja az összefoglaló eszközt, hogy elmentse azokat
3. A régebbi üzenetek biztonságosan eltávolíthatók, mert az összefoglaló rögzíti a fontosakat

Alább definiálunk egy `summarize_preferences` eszközt, amelyet az ügynök meghívhat, hogy tömören rögzítse, mit tanult meg.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Összefoglaló

Ebben a leckében megtanultad, hogyan kezeld a kontextust hosszú lefolyású ügynöki beszélgetések során a Microsoft Agent Framework használatával:

### Kulcsfogalmak
- **A kontextusablakok végesek** — a beszélgetés előzményeiben minden token költséget jelent és beleszámít a korlátba.
- **Összefoglaló eszközök** lehetővé teszik, hogy az ügynök az összegyűjtött kontextust tömör összefoglalóvá alakítsa, csökkentve a tokenhasználatot, miközben megőrzi a lényeges információkat.
- **Ügynöki vázlatlapok** tartós külső memóriát biztosítanak, amelyek túlélnek bármilyen beszélgetés csökkentést.

### Amit építettél
- Egy **kontextusérzékeny ügynököt**, amely fenntartja az folyamatosságot többfordulós beszélgetések során
- Egy **összefoglaló eszközt** (`summarize_preferences`), amely tömör formátumban rögzíti a felhasználó kulcsfontosságú adatait
- Egy **többfordulós beszélgetést**, amely bemutatja a kontextus megőrzését és a változások kezelését

### Valós alkalmazások
- **Ügyfélszolgálati chatbotok**: Megjegyzik a preferenciákat hosszú támogatási munkamenetek alatt
- **Személyi asszisztensek**: Követik a folyamatban lévő projekteket anélkül, hogy újra el kellene magyarázni a kontextust
- **Oktató tutorok**: Fenntartják a tanulói előrehaladást sok interakció során

### Következő lépések
- Valósíts meg egy teljes vázlatlap eszközt fájl alapú tartóssággal
- Adj hozzá automatikus előzménycsökkentést az összefoglaló után
- Kombináld vektorbázisú adattárakkal szemantikus memória kereséshez
- Építs ügynököket, amelyek napokkal később képesek folytatni a beszélgetéseket a teljes kontextussal


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ez a fordítás az alábbi AI fordító szolgáltatással készült: [Co-op Translator](https://github.com/Azure/co-op-translator).** Bár a pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti, anyanyelvi dokumentum tekintendő hivatalos forrásnak. Fontos információk esetén javasolt szakmai, emberi fordítást igénybe venni. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy téves értelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
